In [1]:
import json
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

# Langchain Specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


c:\Users\Nilanjan\Documents\PoCs\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# Step 1: Load and split documents
loader = TextLoader("./3-advanced-semantic-chunking/input_data.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

[Document(metadata={'source': './3-advanced-semantic-chunking/input_data.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains,| agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.')]

In [5]:
# Step 2: FAISS Vector Store Creation qith huggingface embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

c:\Users\Nilanjan\Documents\PoCs\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nilanjan\Documents\PoCs\model_cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4417.86it/s]
BertModel L

In [6]:
# Step 3: Create MMR Retriever
retriever=vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 4, "fetch_k": 20}
)

In [7]:
# Step 4:  Prompt Template and LLM
prompt = PromptTemplate.from_template("""
Answer the following based on the context provided.

Context:
{context}

Question: {input}
Answer:
""")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)


In [9]:
# Step 5: RAG Pipeline
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=documents_chain)

In [10]:
# Step 6: Query
query = {"input": "What are the key features of Langchain?"}
response = rag_chain.invoke(query)

print("Answer:\n", response["answer"])

Answer:
 The key features of Langchain include providing modular abstractions to combine LLMs with tools like OpenAI and Pinecone, the ability to create chains, agents, memory, and retrievers.
